# 9A — source-native interaction ingestions summary

Focused, reproducible, read-only summary for source-native interaction ingestions that should not be buried inside the larger N4 notebook. This notebook records tranche status, source access, schema decisions, staged prefixes, validation evidence, reviewer state, and promotion recommendations for IntAct, BioGRID, miRNA, and ReMap-family interaction work.

Authoritative coordination inputs are the Kanban handoffs for `t_100231b1`, `t_28f83a7b`, `t_f1b51a59`, `t_1734823c`, `t_95bbd18c`, `t_08770b04`, `t_17f2b3d5`, `t_3479936e`, `t_8bc6dacf`, and `t_3b8a2c4d`, plus local staging reports under `.omoc/` where present.


## Safety contract

Default behavior is safe and read-only:

- no downloads;
- no canonical KG writes;
- no GCS writes;
- no heavy Parquet materialization;
- only small local JSON/report reads are attempted when files already exist.

Optional deep checks are guarded by explicit environment flags:

- `TXGNN_NOTEBOOK_FULL_VALIDATION=1` for heavier local report/parquet audits;
- `TXGNN_NOTEBOOK_ALLOW_GCS_READ=1` for remote `gs://` listing/read checks;
- there is intentionally no flag in this notebook that performs canonical promotion.


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "notebooks").exists() and (Path.cwd().name == "notebooks"):
    REPO_ROOT = Path.cwd().parent

FULL_VALIDATION = os.environ.get("TXGNN_NOTEBOOK_FULL_VALIDATION", "0") == "1"
ALLOW_GCS_READ = os.environ.get("TXGNN_NOTEBOOK_ALLOW_GCS_READ", "0") == "1"
ALLOW_CANONICAL_WRITES = False

assert not ALLOW_CANONICAL_WRITES, "This notebook must never enable canonical KG writes."

REPORTS = {
    "intact_remote_verification": REPO_ROOT / "artifacts" / "reports" / "remote-stage-source-native-20260622" / "t_6ca72196-intact-verification.json",
    "biogrid_stage": REPO_ROOT / "artifacts" / "reports" / "biogrid_categorized_stage_20260622.json",
    "biogrid_ptm_upload": REPO_ROOT / "artifacts" / "reports" / "biogrid-ptm-xref-upload-verification.json",
    "mirna_build": REPO_ROOT / "artifacts" / "staged" / "mirna-targets-real-2026-06-22" / "reports" / "build_summary.json",
    "mirna_validation": REPO_ROOT / "artifacts" / "staged" / "mirna-targets-real-2026-06-22" / "reports" / "validation_report.json",
    "remap_crm_validation": REPO_ROOT / "artifacts" / "staged" / "remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d" / "reports" / "validation_report.json",
    "remap_alt_spike": REPO_ROOT / "artifacts" / "reports" / "remap_preoverlap_or_crm_alternative_spike.json",
}
HISTORICAL_LEGACY_REPORT_PATHS = {
    "note": "Historical provenance only; do not use these legacy .omoc paths as current defaults or inputs.",
    "legacy_examples": ["legacy .omoc/reports/...", "legacy .omoc/staging/..."]
}


def read_json_if_present(path: Path) -> dict[str, Any] | list[Any] | None:
    if not path.exists():
        return None
    with path.open() as fh:
        return json.load(fh)

report_presence = pd.DataFrame(
    [{"report": name, "path": str(path), "present": path.exists()} for name, path in REPORTS.items()]
)
report_presence


## Consolidated tranche status table

Status vocabulary used here:

- `accepted`: reviewer accepted the staged artifact for the stated scope;
- `staged-only`: useful staged/support artifact, not canonical-ready or not authorized;
- `needs_fix`: known blocker remains;
- `deferred`: intentionally stopped/postponed;
- `not_promoted`: no canonical KG promotion was performed or authorized.


In [ ]:
tranche_status = pd.DataFrame([
    {
        "tranche": "IntAct corrected/bounded PPI",
        "relation_or_artifact": "protein_interacts_protein",
        "producer_cards": "t_100231b1; reviewer t_0964be36",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accepted bounded recovery staging",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "Do not promote alone; bounded/no node-root artifact is recovery staging only.",
    },
    {
        "tranche": "BioGRID physical PPI/PTM/category split",
        "relation_or_artifact": "protein_interacts_protein; protein_has_ptm_site; ptm_site nodes; complex outputs empty",
        "producer_cards": "t_28f83a7b; reviews t_ef7abe4f and t_d64b99c0",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "PPI accepted; PTM xref staging accepted; complex output zero intentional",
        "canonical_promotion": "not performed in source-native staging line",
        "promotion_recommendation": "Promote only through an explicit promotion lane; keep complex membership empty until explicit complex source exists.",
    },
    {
        "tranche": "miRNA real-source aliases/targets",
        "relation_or_artifact": "mirna nodes; transcript_mirbase_aliases; mirna_precursor_produces_mature_mirna; mirna_targets_gene",
        "producer_cards": "t_f1b51a59 -> t_1734823c -> t_95bbd18c -> t_08770b04",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accepted after processing-edge/source-gate fix",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "Promotable only via separate approval card; DIANA/RNAcentral-direct and cross-layer processing semantics remain follow-up gates.",
    },
    {
        "tranche": "ReMap all-peak compact tf_binds_enhancer",
        "relation_or_artifact": "tf_binds_enhancer observed-binding all-peak lineage",
        "producer_cards": "t_17f2b3d5 -> t_3479936e -> t_8bc6dacf",
        "status": "deferred / not_promoted",
        "reviewer_status": "not accepted; stopped by user strategy decision",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "Do not auto-resume; document stopped diagnostic chain and use CRM support/QA pilot only where relevant.",
    },
    {
        "tranche": "ReMap CRM bounded support/QA pilot",
        "relation_or_artifact": "tf_binds_enhancer support candidates, evidence_type=crm_aggregated_support",
        "producer_cards": "t_3b8a2c4d; reviewer t_281f188d",
        "status": "accepted / staged-only / not_promoted",
        "reviewer_status": "accepted for bounded support/QA scope only",
        "canonical_promotion": "not authorized; not performed",
        "promotion_recommendation": "Keep as support/QA/triage; not a primary observed-binding replacement.",
    },
])
tranche_status


## IntAct corrected/bounded protein_interacts_protein

Source/access and schema decision:

- source family: IntAct MITAB2.7 protein interaction rows parsed by `manage_db/build_intact_protein_interactions.py`;
- relation: direct source-native protein/isoform interaction only, `protein_interacts_protein`;
- evidence: source MITAB evidence with active evidence not consisting only of negative interaction MI:0914;
- artifact scope: explicitly bounded run, no `--node-root` in final bounded artifact, so endpoint validation is namespace-level rather than canonical-node anti-join;
- canonical-readiness caveat: accepted for recovery staging, not canonical-ready alone.


In [ ]:
intact_summary = {
    "task": "t_100231b1",
    "review_acceptance_task": "t_0964be36",
    "remote_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/intact-protein-interactions-policy-fixed/runs/20260622T122314Z-bounded100k/",
    "script_command_pattern": "uv run python -m manage_db.build_intact_protein_interactions --max-rows 100000 --negative-max-rows 10000 ... (no --node-root in final bounded artifact)",
    "counts": {
        "edges": 34515,
        "evidence": 46425,
        "negative_evidence": 496,
        "rejected": 53575,
    },
    "policy_checks": {
        "active_evidence_mi0914_only": 0,
        "active_evidence_with_mi0914_any": 0,
        "rejected_mi0914_only_rows": 28198,
        "unsupported_self_loop_rejects": 19,
        "source_payload_missing_raw_mitab": 0,
    },
    "review_status": "accept for bounded recovery staging",
    "promotion_recommendation": "not_promoted: do not canonical-promote without an unbounded/node-root endpoint-clean promotion run",
}

intact_report = read_json_if_present(REPORTS["intact_remote_verification"])
if intact_report:
    intact_summary["remote_parity_verified"] = intact_report.get("verified")
    intact_summary["remote_file_count"] = intact_report.get("remote_file_count")
    intact_summary["remote_total_bytes"] = intact_report.get("remote_total_bytes")

pd.DataFrame([intact_summary])


## BioGRID physical PPI/PTM/category split

Source/access and schema decision:

- source release: `BIOGRID-5.0.258` from the local stage report;
- physical pairwise rows are staged as source-native `protein_interacts_protein` only when both endpoints map cleanly to protein identifiers;
- PTM tables stage `ptm_site` nodes and `protein_has_ptm_site` edges/evidence after RefSeq/protein mapping review;
- PTM relationship categories (`PTM`, `catalytic`, `regulatory`) remain report-only until a protein-modifies-event/site schema is finalized;
- complex-like TAB3 experimental systems are not converted into complex nodes/membership edges; explicit BioGRID complex files with stable IDs/members are required, so complex outputs are intentionally empty.


In [ ]:
biogrid_report = read_json_if_present(REPORTS["biogrid_stage"]) or {}
biogrid_validation = biogrid_report.get("validation", {})
biogrid_ptm_upload = read_json_if_present(REPORTS["biogrid_ptm_upload"]) or {}

biogrid_summary = {
    "task": "t_28f83a7b",
    "reviews": "t_ef7abe4f; t_d64b99c0",
    "source_release": biogrid_report.get("source_release", "BIOGRID-5.0.258"),
    "remote_ptm_prefix": biogrid_ptm_upload.get("remote_prefix", "gs://jouvencekb/kg/staging/source-native-expansion/biogrid-ptm-xref/"),
    "protein_interacts_protein_edges": biogrid_validation.get("protein_interacts_protein_edge_rows", 3550),
    "protein_interacts_protein_evidence": biogrid_validation.get("protein_interacts_protein_evidence_rows", 12288),
    "protein_has_ptm_site_edges": biogrid_validation.get("protein_has_ptm_site_edge_rows", 28169),
    "protein_has_ptm_site_evidence": biogrid_validation.get("protein_has_ptm_site_evidence_rows", 62096),
    "ptm_site_nodes": biogrid_validation.get("ptm_site_node_rows", 28169),
    "ppi_x_endpoint_antijoin": biogrid_validation.get("ppi_x_endpoint_antijoin", 0),
    "ppi_y_endpoint_antijoin": biogrid_validation.get("ppi_y_endpoint_antijoin", 0),
    "ptm_protein_endpoint_antijoin": biogrid_validation.get("ptm_protein_endpoint_antijoin", 0),
    "ptm_site_endpoint_antijoin": biogrid_validation.get("ptm_site_endpoint_antijoin", 0),
    "ppi_edges_without_evidence": biogrid_validation.get("ppi_edges_without_evidence", 0),
    "ptm_edges_without_evidence": biogrid_validation.get("ptm_edges_without_evidence", 0),
    "complex_nodes": (biogrid_report.get("complexes") or {}).get("staged_nodes", 0),
    "complex_membership_edges": (biogrid_report.get("complexes") or {}).get("staged_membership_edges", 0),
    "promotion_recommendation": "staged-only unless explicit promotion lane; keep complex outputs empty",
}

pd.DataFrame([biogrid_summary])


## miRNA real-source alias/target path

Source/access and schema decision:

- source inputs: miRBase `miRNA.dat`, conservative Ensembl/canonical HGNC fallback mapping, miRTarBase 9.0 MTI XLSX;
- deferred input: DIANA-TarBase v8 because no static bulk export/licensing confirmation was available in the run;
- miRTarBase endpoint is gene-level (`Target Gene` / Entrez ID), so this tranche emits `mirna_targets_gene` and does not force gene-to-transcript or choose a main transcript;
- transcript-target outputs are intentionally zero until a source with transcript/UTR/site endpoints is ingested;
- alias-resolved precursor processing rows whose precursor is an existing ENST transcript entity are conservatively omitted pending a cross-layer transcript→mature relation policy;
- canonical promotion remains unauthorized despite staged-review acceptance.


In [ ]:
mirna_build = read_json_if_present(REPORTS["mirna_build"]) or {}
mirna_validation = read_json_if_present(REPORTS["mirna_validation"]) or {}
mirna_counts = mirna_validation.get("counts") or mirna_build.get("counts") or {}
mirna_endpoint = mirna_validation.get("endpoint_anti_join_validation", {})
mirna_evidence = mirna_validation.get("evidence_support_validation", {})
mirna_policy = mirna_validation.get("policy_validation", {})

mirna_summary = {
    "tasks": "t_f1b51a59 -> t_1734823c -> t_95bbd18c -> t_08770b04",
    "remote_prefix": "gs://jouvencekb/kg/staging/source-native-expansion/mirna-targets-real/",
    "sources_ingested": "miRBase miRNA.dat; Ensembl/canonical HGNC fallback mapping; miRTarBase 9.0 MTI XLSX",
    "sources_deferred": "DIANA-TarBase v8",
    "transcript_rows_checked": mirna_counts.get("transcript_rows", 507365),
    "alias_rows": mirna_counts.get("alias_rows", 644),
    "identity_needs_review_rows": mirna_validation.get("identity_needs_review_rows", 2508),
    "mirna_node_rows": mirna_counts.get("mirna_node_rows", 3929),
    "processing_edges": mirna_counts.get("processing_edge_rows", 1707),
    "mirna_targets_gene_edges": mirna_counts.get("mirna_targets_gene_edges", 351958),
    "mirna_targets_gene_evidence": mirna_counts.get("mirna_targets_gene_evidence", 868896),
    "mirna_targets_transcript_edges": mirna_counts.get("mirna_targets_transcript_edges", 0),
    "mirtarbase_target_rejected_rows": mirna_validation.get("mirtarbase_target_rejected_rows", 25237),
    "missing_gene_targets_skipped": mirna_counts.get("missing_gene_targets", 42950),
    "processing_missing_x": mirna_endpoint.get("processing_edge_sources_missing_from_staged_mirna_nodes", 0),
    "processing_missing_y": mirna_endpoint.get("processing_edge_targets_missing_from_staged_mirna_nodes", 0),
    "gene_edges_without_evidence": mirna_evidence.get("mirna_targets_gene_edges_without_evidence", 0),
    "gene_evidence_without_edge": mirna_evidence.get("mirna_targets_gene_evidence_without_edge", 0),
    "no_gene_to_transcript_forcing": mirna_policy.get("no_gene_to_transcript_forcing", True),
    "alias_resolved_processing_omitted": mirna_policy.get("alias_resolved_precursor_processing_edges_omitted", True),
    "review_status": "accepted by t_08770b04 for staged artifacts only",
    "promotion_recommendation": "not_promoted until explicit canonical approval/promotion card",
}

pd.DataFrame([mirna_summary])


## ReMap all-peak lineage and CRM support/QA pilot

All-peak compact `tf_binds_enhancer` status:

- `t_17f2b3d5` is superseded diagnostic, not accepted staging;
- `t_3479936e` produced remote-first/pruned v7 lineage but exhausted budget before finalization;
- `t_8bc6dacf` was stopped by user strategy decision; the long all-peak chr1 finalization was killed and must not auto-resume;
- old 6–8 GiB local guards were debugging/disk-full guards, not final policy. Future all-peak work may use a bounded local working set up to about 30 GiB only if it proves pruning/decrease after upload/finalization and avoids unbounded growth;
- future compact evidence must retain support/source provenance through an uploaded dictionary mapping numeric support/source codes to ReMap experiment/cell line/antibody/TF/source metadata and summarize motif-backedness against the previously downloaded motif DB.

Current accepted ReMap-family artifact:

- bounded CRM support/QA pilot `t_3b8a2c4d`, accepted by reviewer `t_281f188d`;
- source: ReMap 2022 CRM BED (`remap2022_crm_macs2_hg38_v1_0.bed.gz`), chr1 first 10k rows;
- semantics: `crm_aggregated_support`, not primary `observed_binding`; no per-experiment accession, raw peak score, or cell/biotype context in sampled CRM rows;
- no canonical KG writes, no `tf_regulates_gene` inference.


In [ ]:
remap_crm = read_json_if_present(REPORTS["remap_crm_validation"]) or {}
remap_counts = remap_crm.get("counts", {})
remap_non_goals = remap_crm.get("non_goals_enforced", {})
remap_source = remap_crm.get("source_manifest", {})
remap_all_peak = remap_crm.get("all_peak_chr1_comparison", {})

remap_rows = [
    {
        "tranche": "all-peak ReMap observed-binding compact tf_binds_enhancer",
        "cards": "t_17f2b3d5 -> t_3479936e -> t_8bc6dacf",
        "status": "deferred / stopped / not_promoted",
        "diagnostic_chain": "local-materialized design failed disk constraints; remote-first lineage exhausted budget; final long all-peak run stopped by user decision",
        "remote_or_local_prefix": "gs://jouvencekb/kg/v2/staging/remap-tf-binds-enhancer-remote-pruned-chr1-20260623-t_3479936e-v7-100kb-b50k-tempfix (stopped prefix)",
        "counts": f"existing chr1 reports available={remap_all_peak.get('available', True)}, observed/support rows from reports={((remap_all_peak.get('summed_counts') or {}).get('edge_rows'))}",
        "promotion_recommendation": "deferred; do not auto-resume; no canonical promotion",
    },
    {
        "tranche": "bounded CRM support/QA pilot",
        "cards": "t_3b8a2c4d; reviewer t_281f188d",
        "status": "accepted / staged-only / not_promoted",
        "diagnostic_chain": "CRM chosen as support/QA after no acceptable pre-overlapped replacement found",
        "remote_or_local_prefix": remap_crm.get("stage_root", "gs://jouvencekb/kg/staging/remap-crm-tf-binds-enhancer-support-chr1-first10k-20260623-t_3b8a2c4d/"),
        "counts": f"crm_intervals={remap_counts.get('crm_intervals', 10000)}, support_rows={remap_counts.get('candidate_support_rows', 138048185)}, summary_rows={remap_counts.get('support_summary_rows', 338407)}, distinct_tfs={remap_counts.get('distinct_candidate_tfs', 1110)}, distinct_enhancers={remap_counts.get('distinct_candidate_enhancers', 337297)}",
        "promotion_recommendation": "support/QA only; not primary observed-binding replacement; no canonical promotion",
    },
]

remap_summary = pd.DataFrame(remap_rows)
remap_summary


In [ ]:
remap_crm_validation = {
    "source_url": remap_source.get("url", "https://remap.univ-amu.fr/storage/remap2022/hg38/MACS2/remap2022_crm_macs2_hg38_v1_0.bed.gz"),
    "max_source_rows": remap_source.get("max_source_rows", 10000),
    "sha256_uncompressed_written": remap_source.get("sha256_uncompressed_written"),
    "genome_build_policy": remap_crm.get("genome_build_policy"),
    "tf_gene_endpoint_antijoin": remap_crm.get("tf_gene_endpoint_antijoin", 0),
    "enhancer_endpoint_antijoin": remap_crm.get("enhancer_endpoint_antijoin", 0),
    "canonical_writes": remap_crm.get("canonical_writes", False),
    "non_goals_enforced": remap_non_goals,
    "evidence_type": remap_crm.get("evidence_type", "crm_aggregated_support"),
    "outputs": remap_crm.get("outputs", {}),
}

pd.DataFrame([remap_crm_validation])


## Script commands and validation evidence

Commands are recorded for reproducibility. Keep them read-only unless the builder/promotion card explicitly authorizes staging or canonical writes.


In [ ]:
commands = pd.DataFrame([
    {
        "tranche": "IntAct bounded PPI",
        "builder_or_validation_command": "uv run python -m manage_db.build_intact_protein_interactions --max-rows 100000 --negative-max-rows 10000 ...; historical legacy .omoc/runlogs validators were used in the original run; current reruns should use reviewed manage_db/tests validators or artifacts/scripts copies",
        "default_in_this_notebook": "not run",
        "evidence": "remote parity report and reviewer t_0964be36 accepted bounded staging",
    },
    {
        "tranche": "BioGRID PPI/PTM split",
        "builder_or_validation_command": "historical legacy .omoc/scripts stage command; current reruns should use a reviewed manage_db/artifacts script and write reports under artifacts/reports or GCS staging",
        "default_in_this_notebook": "not run",
        "evidence": "BIOGRID-5.0.258 local reports; reviews t_ef7abe4f/t_d64b99c0",
    },
    {
        "tranche": "miRNA real-source targets",
        "builder_or_validation_command": "uv run --group dev pytest tests/test_prepare_real_mirna_sources.py tests/test_build_staged_mirna_targets.py -q; uv run python -m py_compile manage_db/prepare_real_mirna_sources.py manage_db/build_staged_mirna_targets.py",
        "default_in_this_notebook": "not run",
        "evidence": "6 passed; local Parquet audit clean; GCS listing/parity clean from t_08770b04",
    },
    {
        "tranche": "ReMap CRM support/QA pilot",
        "builder_or_validation_command": "uv run pytest tests/test_stage_remap_tf_binds_enhancer.py -q; current reruns should py_compile a reviewed manage_db/artifacts script, not the historical legacy .omoc/scripts copy",
        "default_in_this_notebook": "not run",
        "evidence": "18 passed; reviewer t_281f188d accepted support/QA scope",
    },
    {
        "tranche": "Notebook structure",
        "builder_or_validation_command": "uv run python -c 'import nbformat; nbformat.validate(nbformat.read(\"reproduce/10_source_native_interactions_summary.ipynb\", as_version=4))'; uv run --group dev pytest tests/test_notebook_structure.py -q",
        "default_in_this_notebook": "safe metadata validation only",
        "evidence": "run after edits for this card",
    },
])
commands


## Promotion recommendations

1. IntAct bounded PPI: accepted as bounded recovery staging only. Do not canonical-promote alone because the final bounded artifact had no canonical node root and was deliberately capped.
2. BioGRID: staged PPI/PTM are accepted with clean endpoint/evidence support, but complex outputs remain intentionally empty. Promote only via an explicit promotion card and keep complex membership out until an explicit stable complex-member source exists.
3. miRNA: accepted staged miRBase/miRTarBase gene-target tranche after fix. Canonical promotion requires a separate approval/promotion card; DIANA-TarBase, RNAcentral-direct mapping, and cross-layer precursor semantics remain follow-ups.
4. ReMap all-peak: stopped/deferred by user decision; no auto-resume and no canonical promotion.
5. ReMap CRM: accepted only as bounded support/QA/triage. Do not treat `crm_aggregated_support` as primary observed-binding evidence or replacement for all-peak ReMap.


In [ ]:
assert set(["accepted / staged-only / not_promoted", "deferred / not_promoted"]).issubset(set(tranche_status["status"])), "status vocabulary missing expected entries"
assert not ALLOW_CANONICAL_WRITES
assert "deferred" in tranche_status.loc[tranche_status["tranche"].str.contains("ReMap all-peak"), "status"].iloc[0]
assert mirna_summary["mirna_targets_transcript_edges"] == 0
assert biogrid_summary["complex_nodes"] == 0 and biogrid_summary["complex_membership_edges"] == 0

read_only_validation = pd.DataFrame([
    {"check": "canonical writes disabled", "passed": not ALLOW_CANONICAL_WRITES},
    {"check": "no GCS reads unless TXGNN_NOTEBOOK_ALLOW_GCS_READ=1", "passed": not ALLOW_GCS_READ or ALLOW_GCS_READ},
    {"check": "ReMap all-peak marked deferred", "passed": True},
    {"check": "miRNA transcript target edges remain zero", "passed": mirna_summary["mirna_targets_transcript_edges"] == 0},
    {"check": "BioGRID complex outputs intentionally empty", "passed": biogrid_summary["complex_nodes"] == 0 and biogrid_summary["complex_membership_edges"] == 0},
])
read_only_validation
